# Manual Event Annotation & Extraction Evaluation

Blind manual annotation of event triggers + MAVEN event types over the **final evaluation corpus**
(`experiment.jsonl`, 5,666 summaries), and MAVEN-style **trigger-identification** and **classification**
F1 for the BERT+CRF extractor — the Tell Me Again! analogue of its 67.3 MAVEN F1.
Spec: `docs/superpowers/specs/2026-06-09-manual-event-annotation-design.md`.

## Cells (run top-to-bottom)
1. **Setup** — imports, paths, constants (`VARIANT`, `ANNOTATOR`, `SEED`, `MIN_SENT`/`MAX_SENT`, `N_TARGET`).
2. **Slim corpus cache** — streams the 4 GB `experiment.jsonl` to a small `gold/annotation_corpus.jsonl`.
3. **Storage helpers** — resumable, atomic gold + manifest writes.
4. **Candidates + resume** — deterministic seeded sample of 5–10-sentence summaries; resumes where you left off.
5. **Types + FrameNet guidance** — 168 types with FrameNet **definitions** + lexical-unit lookup
   (the ontology codebook — *not* MAVEN-train frequencies — so guidance doesn't bias the gold toward the model).
6. **Annotation widget** — blind UI: pick sentence → trigger word(s) → search/scroll the 168 types → *Add*.
7. **Evaluation** — trigger-ID + classification P/R/F1 (exact + overlap), per-type + confusion → `gold/annotation_eval.<variant>.yaml`.
8. **Inspection** — random examples per bucket (correct / type-wrong / missed / spurious) to sanity-check.

## How to annotate
- **Exhaustively**: mark *every* event trigger in each summary before *Next →* (unmarked real events count as
  model false positives and wreck precision).
- **Blind**: the widget never shows the model's predictions.
- Autosaves to `gold/manual_events.<variant>.jsonl`; only summaries with ≥1 event are stored.

## Resume / extend / anonymized variant
- **Resume**: just re-run top-to-bottom in any later session — done summaries are skipped.
- **Annotate more**: raise `N_TARGET`.
- **Anonymized variant**: set `VARIANT="anon"`, point `EXPERIMENT_DIR` at the anon run, and pass the
  cross-corpus id intersection to `select_candidates(restrict_to_ids=…)`.

> Optional terminal aid (separate, MAVEN-frequency-based — keep out of the unbiased protocol unless your
> supervisor approves): `python trigger_lookup.py`.


In [12]:
# === Manual event-annotation: setup & config ===
import json, random, collections, re, sys
from datetime import datetime
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, Markdown
import yaml
import pandas as pd

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "models" / "bert_crf"))   # for utils_maven.get_labels

# --- which corpus + who is annotating (anon = config flip later) ----------------------
EXPERIMENT_DIR = ROOT / "data" / "experiments" / "experiment_test_9385_20260609_1013"
VARIANT   = "non_anon"          # "non_anon" | "anon"
ANNOTATOR = "bugra"
SEED      = 20260609
MIN_SENT, MAX_SENT = 5, 10
N_TARGET  = 30                  # raise later to annotate more; continues the same order

CORPUS_PATH   = EXPERIMENT_DIR / "experiment.jsonl"
GOLD_DIR      = EXPERIMENT_DIR / "gold"; GOLD_DIR.mkdir(parents=True, exist_ok=True)
SLIM_PATH     = GOLD_DIR / "annotation_corpus.jsonl"
GOLD_PATH     = GOLD_DIR / f"manual_events.{VARIANT}.jsonl"
MANIFEST_PATH = GOLD_DIR / "annotation_progress.json"
EVAL_PATH     = GOLD_DIR / f"annotation_eval.{VARIANT}.yaml"
TYPE_EX_PATH  = ROOT / "data" / "intermediate" / "maven_type_examples.json"
MAVEN_TRAIN   = ROOT / "data" / "raw" / "MAVEN" / "train.jsonl"

KEEP_FIELDS = ("wikidata_id","summary_id","lang","text","sentences","n_sentences","events")
def uid_of(r): return f'{r["wikidata_id"]}__{r["summary_id"]}'   # composite key (summary_id is a lang code)

print("config OK:", EXPERIMENT_DIR.name, "| variant:", VARIANT, "| annotator:", ANNOTATOR)


config OK: experiment_test_9385_20260609_1013 | variant: non_anon | annotator: bugra


In [2]:
# === Build-or-load the slim corpus cache (drops the 4GB of embeddings/baselines) ===
def build_slim_corpus(src: Path, dst: Path, keep=KEEP_FIELDS) -> int:
    n = 0
    tmp = dst.with_suffix(dst.suffix + ".tmp")
    with src.open(encoding="utf-8") as fin, tmp.open("w", encoding="utf-8") as fout:
        for line in fin:
            if not line.strip(): continue
            r = json.loads(line)
            fout.write(json.dumps({k: r[k] for k in keep}, ensure_ascii=False) + "\n")
            n += 1
    tmp.replace(dst)
    return n

if not SLIM_PATH.exists():
    print(f"Building slim cache from {CORPUS_PATH.name} (one-time, ~1-2 min)...")
    print(f"  wrote {build_slim_corpus(CORPUS_PATH, SLIM_PATH)} rows -> {SLIM_PATH.name}")

corpus = [json.loads(l) for l in SLIM_PATH.read_text(encoding="utf-8").splitlines() if l.strip()]
by_uid = {uid_of(r): r for r in corpus}
print(f"loaded slim corpus: {len(corpus)} summaries ({len(by_uid)} unique uids)")


loaded slim corpus: 5666 summaries (5666 unique uids)


In [3]:
# === Gold + manifest persistence (uid-keyed; atomic; resumable across sessions) ===
def load_gold(path: Path) -> dict:
    if not path.exists(): return {}
    out = {}
    for line in path.read_text(encoding="utf-8").splitlines():
        if not line.strip(): continue
        r = json.loads(line); out[f'{r["wikidata_id"]}__{r["summary_id"]}'] = r
    return out

def save_gold(path: Path, gold: dict) -> None:
    tmp = path.with_suffix(path.suffix + ".tmp")
    with tmp.open("w", encoding="utf-8") as f:
        for r in gold.values():
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    tmp.replace(path)

def update_manifest(path: Path, gold: dict) -> None:
    doc = json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}
    doc.setdefault("config", {"seed": SEED, "min_sent": MIN_SENT, "max_sent": MAX_SENT})
    doc.setdefault("annotators", {})[ANNOTATOR] = {
        "variant": VARIANT, "count": len(gold), "done_uids": sorted(gold.keys()),
    }
    path.write_text(json.dumps(doc, indent=2, ensure_ascii=False), encoding="utf-8")

def gold_record(uid: str, events: list) -> dict:
    r = by_uid[uid]
    return {"annotator": ANNOTATOR, "variant": VARIANT,
            "wikidata_id": r["wikidata_id"], "summary_id": r["summary_id"],
            "events": events, "annotated_at": datetime.now().isoformat(timespec="seconds")}

print("storage helpers defined")


storage helpers defined


In [4]:
# === Deterministic seeded candidate order + resume state ===
def select_candidates(corpus, min_sent, max_sent, seed, restrict_to_ids=None):
    """Seeded order of candidate uids filtered by sentence count.
    restrict_to_ids: optional set of uids to keep (e.g. intersection with the anon corpus)."""
    pool = [uid_of(r) for r in corpus if min_sent <= r["n_sentences"] <= max_sent]
    if restrict_to_ids is not None:
        keep = set(restrict_to_ids); pool = [u for u in pool if u in keep]
    pool.sort()                              # stable base order before shuffle
    random.Random(seed).shuffle(pool)
    return pool

candidates = select_candidates(corpus, MIN_SENT, MAX_SENT, SEED)
gold = load_gold(GOLD_PATH)                  # uid -> record (resume)
done = set(gold.keys())
remaining = [u for u in candidates if u not in done]
print(f"{len(candidates)} candidates ({MIN_SENT}-{MAX_SENT} sentences) | "
      f"annotated {len(done)} | target N={N_TARGET} | next up: {len(remaining)} remaining")


1529 candidates (5-10 sentences) | annotated 0 | target N=30 | next up: 1529 remaining


In [5]:
# === 168 MAVEN types + FrameNet codebook guidance (definitions + lexical-unit index) ===
# Guidance is grounded in the FrameNet ontology (the codebook MAVEN is built on), NOT the MAVEN training
# annotations — so it informs correct ontology use WITHOUT leaking the model's training signal into the
# gold standard. 144/168 types map to a FrameNet frame (definition + lexical units); the other 24 are
# MAVEN-renamed types shown name-only.
from utils_maven import get_labels
from nltk.stem import WordNetLemmatizer

FN_DEF_PATH = ROOT / "data" / "intermediate" / "framenet_definitions.json"   # {type: definition|None}
FN_LU_PATH  = ROOT / "data" / "intermediate" / "framenet_lu_index.json"      # {lemma: [maven_types]}

def maven_types():
    return sorted({lbl[2:] for lbl in get_labels("") if lbl != "O"})

def build_framenet_guidance(types, def_out: Path, lu_out: Path):
    import nltk
    try:
        from nltk.corpus import framenet as fn; fn.frames()
    except LookupError:
        print("Downloading FrameNet (framenet_v17, one-time)...")
        nltk.download("framenet_v17", quiet=True)
        from nltk.corpus import framenet as fn
    fn_by_name = {f.name: f for f in fn.frames()}
    strip = lambda s: re.sub(r"\s+", " ", re.sub(r"<[^>]+>", "", s)).strip()
    defs, lu_index = {}, collections.defaultdict(set)
    for t in types:
        fr = fn_by_name.get(t)
        defs[t] = strip(fr.definition) if fr is not None else None
        if fr is not None:
            for lu in fr.lexUnit.keys():                       # e.g. "attack.v"
                lu_index[lu.rsplit(".", 1)[0].lower()].add(t)  # MAVEN-type frames only
    lu_index = {w: sorted(ts) for w, ts in lu_index.items()}
    def_out.parent.mkdir(parents=True, exist_ok=True)
    def_out.write_text(json.dumps(defs, ensure_ascii=False), encoding="utf-8")
    lu_out.write_text(json.dumps(lu_index, ensure_ascii=False), encoding="utf-8")
    return defs, lu_index

TYPES = maven_types(); TYPE_SET = set(TYPES)
if not (FN_DEF_PATH.exists() and FN_LU_PATH.exists()):
    build_framenet_guidance(TYPES, FN_DEF_PATH, FN_LU_PATH)
TYPE_DEF = json.loads(FN_DEF_PATH.read_text(encoding="utf-8"))    # {type: definition|None}
LU_INDEX = json.loads(FN_LU_PATH.read_text(encoding="utf-8"))     # {lemma: [maven_types]}

# Lemma-aware lookup: FrameNet LUs are lemmas, but annotators select inflected surface words.
import nltk
try: nltk.data.find("corpora/wordnet")
except LookupError: nltk.download("wordnet", quiet=True)
_LEMM = WordNetLemmatizer()
def frames_for(word):
    """MAVEN frames listing `word` (or its verb/noun lemma) as a FrameNet lexical unit — definitional."""
    w = word.lower().strip(); out = []
    for c in (w, _LEMM.lemmatize(w, "v"), _LEMM.lemmatize(w, "n")):
        for t in LU_INDEX.get(c, []):
            if t not in out: out.append(t)
    return out

n_def = sum(1 for t in TYPES if TYPE_DEF.get(t))
print(f"{len(TYPES)} types | {n_def} with FrameNet definitions | {len(LU_INDEX)} lexical units | "
      f"'attacked' -> {frames_for('attacked')}")


168 types | 144 with FrameNet definitions | 1968 lexical units | 'attacked' -> ['Attack', 'Judgment_communication']


In [6]:
# === Blind annotation widget ===
WORD_RE = re.compile(r"[A-Za-z0-9]+(?:['-][A-Za-z0-9]+)*")

def tokenize(sentence):
    """[(idx, token_text, start, end), ...] — word spans excluding punctuation."""
    return [(i, m.group(0), m.start(), m.end()) for i, m in enumerate(WORD_RE.finditer(sentence))]

class Annotator:
    def __init__(self, uids):
        self.uids = uids
        self.cur = next((i for i, u in enumerate(uids) if u not in gold), 0)
        self.events = []                       # working events for the current summary
        self._build()
        self._load_current()
        display(self.ui)

    def _build(self):
        self._syncing = False                  # reentrancy guard for word_sel <-> lookup sync
        self.progress = widgets.HTML()
        self.text     = widgets.HTML()
        self.sent_dd  = widgets.Dropdown(description="Sentence:", layout=widgets.Layout(width="95%"))
        self.word_sel = widgets.SelectMultiple(description="Trigger:", rows=6,
                                               layout=widgets.Layout(width="60%"))
        self.trig_lookup = widgets.Combobox(description="Lookup:", ensure_option=False, options=(),
                                            placeholder="type a sentence word, Enter",
                                            layout=widgets.Layout(width="60%"))
        self.suggest  = widgets.HTML()
        # Type picker: a filter box + a SCROLLABLE listbox of all 168 types ("" = none).
        self.type_search = widgets.Text(description="Type:", placeholder="filter 168 types…",
                                        layout=widgets.Layout(width="60%"))
        self.type_list = widgets.Select(options=("",) + tuple(TYPES), value="", rows=10,
                                        description="", layout=widgets.Layout(width="60%"))
        self.type_hint = widgets.HTML()
        self.add_btn  = widgets.Button(description="Add event trigger and Event category",
                                       button_style="success", layout=widgets.Layout(width="320px"))
        self.events_box = widgets.VBox()
        self.prev_btn = widgets.Button(description="← Prev")
        self.next_btn = widgets.Button(description="Next →", button_style="primary")
        self.msg = widgets.HTML()

        self.sent_dd.observe(self._on_sent, names="value")
        self.word_sel.observe(self._on_words, names="value")
        self.trig_lookup.observe(self._on_lookup, names="value")
        self.type_search.observe(self._on_type_search, names="value")
        self.type_list.observe(self._on_type, names="value")
        self.add_btn.on_click(self._on_add)
        self.prev_btn.on_click(lambda b: self._go(-1))
        self.next_btn.on_click(lambda b: self._go(+1))
        self.ui = widgets.VBox([
            self.progress, self.text,
            self.sent_dd, self.word_sel,
            self.trig_lookup, self.suggest,
            self.type_search, self.type_list, self.type_hint,
            self.add_btn, widgets.HTML("<b>Events:</b>"), self.events_box,
            widgets.HBox([self.prev_btn, self.next_btn]), self.msg,
        ])

    @property
    def row(self): return by_uid[self.uids[self.cur]]

    def _reset_type(self):
        self.type_search.value = ""
        self.type_list.options = ("",) + tuple(TYPES)
        self.type_list.value = ""
        self.type_hint.value = ""

    def _load_current(self):
        u = self.uids[self.cur]
        self.events = [dict(e) for e in gold[u]["events"]] if u in gold else []
        sents = self.row["sentences"]
        self.sent_dd.options = [(f"{i}: {s[:60]}", i) for i, s in enumerate(sents)]
        self.sent_dd.value = 0
        self._on_sent(None); self._reset_type(); self._render()

    def _render(self):
        u = self.uids[self.cur]
        self.progress.value = (f"<b>uid {self.cur+1}/{len(self.uids)}</b> &nbsp; {u} &nbsp; "
                               f"(annotated {len(gold)}, target {N_TARGET})"
                               + ("  ✅ done" if u in gold else ""))
        self.text.value = "<br>".join(f"<b>{i}.</b> {s}" for i, s in enumerate(self.row["sentences"]))
        rows = []
        for k, e in enumerate(self.events):
            lbl = widgets.HTML(f"s{e['sent_id']} &nbsp; <code>{e['trigger']}</code> "
                               f"&rarr; <b>{e['event_type']}</b>")
            x = widgets.Button(description="✕", layout=widgets.Layout(width="34px"))
            x.on_click(lambda b, idx=k: self._del(idx))
            rows.append(widgets.HBox([x, lbl]))
        self.events_box.children = rows

    def _selected_surface(self):
        sel = sorted(self.word_sel.value)
        if self.sent_dd.value is None or not sel: return None
        toks = tokenize(self.row["sentences"][self.sent_dd.value])
        return self.row["sentences"][self.sent_dd.value][toks[sel[0]][2]:toks[sel[-1]][3]]

    def _show_suggest(self, word):
        w = (word or "").lower().strip()
        frames = frames_for(w) if w else []
        if not w:       self.suggest.value = ""
        elif frames:    self.suggest.value = "📖 lexical unit of: " + ", ".join(f"<b>{t}</b>" for t in frames)
        else:           self.suggest.value = (f"<i>no FrameNet frame lists &ldquo;{w}&rdquo; — use the "
                                              f"Type filter + definitions</i>")

    def _on_sent(self, _):
        if self.sent_dd.value is None: return
        toks = tokenize(self.row["sentences"][self.sent_dd.value])
        self._syncing = True
        self.word_sel.options = [(f"{i}: {t}", i) for i, t, s, e in toks]
        self.word_sel.value = ()
        self.trig_lookup.options = tuple(dict.fromkeys(t for _, t, s, e in toks))   # unique sentence words
        self.trig_lookup.value = ""
        self._syncing = False
        self.suggest.value = ""

    def _on_words(self, _):
        if self._syncing: return
        surface = self._selected_surface()
        if surface is None: return
        self._syncing = True; self.trig_lookup.value = surface; self._syncing = False
        self._show_suggest(surface)

    def _on_lookup(self, _):
        self._show_suggest(self.trig_lookup.value)
        if self._syncing or self.sent_dd.value is None: return
        w = self.trig_lookup.value.lower().strip()
        toks = tokenize(self.row["sentences"][self.sent_dd.value])
        matches = [i for i, t, s, e in toks if t.lower() == w]
        if matches:                                  # map typed word -> first matching token span
            self._syncing = True; self.word_sel.value = (matches[0],); self._syncing = False

    def _on_type_search(self, _):
        q = self.type_search.value.lower().strip()
        opts = [t for t in TYPES if q in t.lower()] if q else list(TYPES)
        self.type_list.options = ("",) + tuple(opts)
        self.type_list.value = ""

    def _on_type(self, _):
        t = self.type_list.value
        if t in TYPE_SET:
            d = TYPE_DEF.get(t) or "(no FrameNet definition — MAVEN-specific type; apply by judgment)"
            self.type_hint.value = f"<b>FrameNet Explanation:</b> <i>{d}</i>"
        else:
            self.type_hint.value = ""

    def _on_add(self, _):
        sel = sorted(self.word_sel.value); etype = self.type_list.value
        if not sel:        self.msg.value = "<span style='color:red'>select trigger word(s)</span>"; return
        if etype not in TYPE_SET: self.msg.value = "<span style='color:red'>pick a valid type</span>"; return
        toks = tokenize(self.row["sentences"][self.sent_dd.value])
        start = toks[sel[0]][2]; end = toks[sel[-1]][3]
        trigger = self.row["sentences"][self.sent_dd.value][start:end]
        self.events.append({"event_id": f"g{len(self.events)+1}", "sent_id": self.sent_dd.value,
                            "trigger": trigger, "event_type": etype, "start": start, "end": end})
        self._syncing = True
        self.word_sel.value = (); self.trig_lookup.value = ""
        self._syncing = False
        self.suggest.value = ""; self.msg.value = ""; self._reset_type()
        self._autosave(); self._render()

    def _del(self, idx):
        self.events.pop(idx)
        for k, e in enumerate(self.events): e["event_id"] = f"g{k+1}"
        self._autosave(); self._render()

    def _autosave(self):
        u = self.uids[self.cur]
        if self.events:                                  # only persist summaries with >=1 event
            gold[u] = gold_record(u, [dict(e) for e in self.events])
        elif u in gold:
            del gold[u]                                  # drop an emptied/phantom record
        save_gold(GOLD_PATH, gold); update_manifest(MANIFEST_PATH, gold)

    def _go(self, step):
        self._autosave()
        self.cur = max(0, min(len(self.uids) - 1, self.cur + step))
        self._load_current()

annotator = Annotator(candidates)


In [7]:
# === MAVEN-style evaluation: gold (manual) vs predicted (BERT+CRF) ===
# Match gold<->predicted events per summary on sent_id + span; report micro P/R/F1 for trigger
# Identification (span only) and Classification (span AND type), plus per-type and a type-confusion
# view. Exact span = headline (MAVEN-style); overlap = lenient. Written to gold/annotation_eval.<variant>.yaml.
def match_pairs(gold_evs, pred_evs, mode="exact"):
    """One-to-one (gold_idx, pred_idx) span matches: same sent_id + (exact span | best overlap)."""
    used, pairs = set(), []
    for gi, g in enumerate(gold_evs):
        best = None
        for pj, p in enumerate(pred_evs):
            if pj in used or p["sent_id"] != g["sent_id"]: continue
            if mode == "exact":
                score = 1 if (p["start"], p["end"]) == (g["start"], g["end"]) else 0
            else:
                score = max(0, min(p["end"], g["end"]) - max(p["start"], g["start"]))
            if score > 0 and (best is None or score > best[0]): best = (score, pj)
        if best: used.add(best[1]); pairs.append((gi, best[1]))
    return pairs

def _prf(tp, n_pred, n_gold):
    P = tp / n_pred if n_pred else 0.0
    R = tp / n_gold if n_gold else 0.0
    F = 2 * P * R / (P + R) if (P + R) else 0.0
    return {"P": round(P, 4), "R": round(R, 4), "F1": round(F, 4)}

def evaluate(gold, by_uid, mode="exact"):
    ID_tp = CLS_tp = NP = NG = 0
    pt_tp, pt_pred, pt_gold = collections.Counter(), collections.Counter(), collections.Counter()
    confusion = collections.Counter(); n_empty = 0
    for u, rec in gold.items():
        G = rec["events"]; P = by_uid.get(u, {"events": []})["events"]
        if not G: n_empty += 1
        NG += len(G); NP += len(P)
        for e in G: pt_gold[e["event_type"]] += 1
        for e in P: pt_pred[e["event_type"]] += 1
        for gi, pj in match_pairs(G, P, mode):
            ID_tp += 1
            gt, ptp = G[gi]["event_type"], P[pj]["event_type"]
            if gt == ptp: CLS_tp += 1; pt_tp[gt] += 1
            else: confusion[(gt, ptp)] += 1
    per_type = {t: {"support": pt_gold[t], **_prf(pt_tp[t], pt_pred[t], pt_gold[t])}
                for t in sorted(pt_gold)}
    return {"mode": mode, "n_summaries": len(gold), "n_summaries_no_gold_events": n_empty,
            "n_gold": NG, "n_pred": NP,
            "trigger_identification": _prf(ID_tp, NP, NG),
            "trigger_classification": _prf(CLS_tp, NP, NG),
            "per_type": per_type,
            "top_confusions": [{"gold": g, "pred": p, "n": n} for (g, p), n in confusion.most_common(15)]}

results = {m: evaluate(gold, by_uid, mode=m) for m in ("exact", "overlap")}
EVAL_PATH.write_text(yaml.safe_dump(results, sort_keys=False, allow_unicode=True), encoding="utf-8")

for m in ("exact", "overlap"):
    r = results[m]
    print(f"[{m}]  summaries={r['n_summaries']} (no-gold-events: {r['n_summaries_no_gold_events']})  "
          f"gold={r['n_gold']}  pred={r['n_pred']}")
    print(f"   Trigger ID : {r['trigger_identification']}")
    print(f"   Trigger CLS: {r['trigger_classification']}")
print(f"\nwrote {EVAL_PATH.name}")


[exact]  summaries=0 (no-gold-events: 0)  gold=0  pred=0
   Trigger ID : {'P': 0.0, 'R': 0.0, 'F1': 0.0}
   Trigger CLS: {'P': 0.0, 'R': 0.0, 'F1': 0.0}
[overlap]  summaries=0 (no-gold-events: 0)  gold=0  pred=0
   Trigger ID : {'P': 0.0, 'R': 0.0, 'F1': 0.0}
   Trigger CLS: {'P': 0.0, 'R': 0.0, 'F1': 0.0}

wrote annotation_eval.non_anon.yaml


In [10]:
# === Sanity check: gold (manual) vs BERT+CRF, sampled across match/category buckets ===
# Buckets (exact-span matching): correct (span+type) | span match but type wrong | missed by model (FN)
# | spurious model prediction (FP). Shows N_PER random examples per bucket with the trigger 【marked】.
# Run after the evaluation cell (reuses match_pairs). Change SEED_INSPECT to resample.
import random
N_PER = 2
SEED_INSPECT = 0

def _classify(gold, by_uid, mode="exact"):
    B = {"correct (span+type)": [], "span match, type WRONG": [],
         "missed by model (FN)": [], "spurious model pred (FP)": []}
    for u, rec in gold.items():
        G = rec["events"]; row = by_uid.get(u, {"events": [], "sentences": []})
        P, sents = row["events"], row["sentences"]
        def sent(i): return sents[i] if i < len(sents) else ""
        pairs = match_pairs(G, P, mode)
        mg = {gi for gi, _ in pairs}; mp = {pj for _, pj in pairs}
        for gi, pj in pairs:
            g, p = G[gi], P[pj]
            key = "correct (span+type)" if g["event_type"] == p["event_type"] else "span match, type WRONG"
            B[key].append({"uid": u, "sent": sent(g["sent_id"]), "span": (g["start"], g["end"]),
                           "g": (g["trigger"], g["event_type"]), "p": (p["trigger"], p["event_type"])})
        for gi, g in enumerate(G):
            if gi not in mg:
                B["missed by model (FN)"].append({"uid": u, "sent": sent(g["sent_id"]),
                    "span": (g["start"], g["end"]), "g": (g["trigger"], g["event_type"]), "p": None})
        for pj, p in enumerate(P):
            if pj not in mp:
                B["spurious model pred (FP)"].append({"uid": u, "sent": sent(p["sent_id"]),
                    "span": (p["start"], p["end"]), "g": None, "p": (p["trigger"], p["event_type"])})
    return B

def _mark(s, span):
    a, b = span; return s[:a] + "【" + s[a:b] + "】" + s[b:]

_B = _classify(gold, by_uid, "exact")
_rng = random.Random(SEED_INSPECT)
_out = ["## Gold vs BERT+CRF — random examples per bucket (exact-span)\n"]
for name, items in _B.items():
    _out.append(f"### {name} — {len(items)} total")
    if not items:
        _out.append("_(none)_\n"); continue
    for ex in _rng.sample(items, min(N_PER, len(items))):
        _out.append(f"- `{ex['uid']}` — {_mark(ex['sent'], ex['span'])}")
        if ex["g"]: _out.append(f"    - **gold**: `{ex['g'][0]}` → **{ex['g'][1]}**")
        if ex["p"]: _out.append(f"    - **bert+crf**: `{ex['p'][0]}` → **{ex['p'][1]}**")
    _out.append("")
display(Markdown("\n".join(_out)))


## Gold vs BERT+CRF — random examples per bucket (exact-span)

### correct (span+type) — 0 total
_(none)_

### span match, type WRONG — 0 total
_(none)_

### missed by model (FN) — 0 total
_(none)_

### spurious model pred (FP) — 0 total
_(none)_


In [15]:
# === Thesis table: BERT+CRF trigger-extraction quality vs the manual gold ===
# MAVEN's standard metric (Wang et al., 2020) is MICRO, SPAN-BASED Precision/Recall/F1 for trigger
# CLASSIFICATION (span + type). We report that plus trigger IDENTIFICATION (span only) and Macro-F1
# over types, for exact- and overlap-span matching. Values in %. Reuses `results` from the eval cell.
# The MAVEN row is an in-domain, candidate-based reference (not a like-for-like comparison) — see Note.
# MAVEN BERT+CRF (Table 5): P 65.0 / R 70.9 / F1 67.8 ± 0.15 (micro, span-based, mean over 10 runs).
def _macro_f1(per_type):
    fs = [v["F1"] for v in per_type.values() if v["support"] > 0]
    return round(100 * sum(fs) / len(fs), 1) if fs else None

def _pct(x): return round(100 * x, 1)

rows = []
for mode in ("exact", "overlap"):
    r = results[mode]
    ti, tc = r["trigger_identification"], r["trigger_classification"]
    rows.append({"Dataset": "TMA", "Evaluation": "Trigger identification", "Span match": mode,
                 "P": _pct(ti["P"]), "R": _pct(ti["R"]), "F1": _pct(ti["F1"]),
                 "Macro-F1": None, "n_gold": r["n_gold"], "n_pred": r["n_pred"]})
    rows.append({"Dataset": "TMA", "Evaluation": "Trigger classification", "Span match": mode,
                 "P": _pct(tc["P"]), "R": _pct(tc["R"]), "F1": _pct(tc["F1"]),
                 "Macro-F1": _macro_f1(r["per_type"]), "n_gold": r["n_gold"], "n_pred": r["n_pred"]})
rows.append({"Dataset": "MAVEN", "Evaluation": "Trigger classification", "Span match": "exact",
             "P": 65.0, "R": 70.9, "F1": 67.8, "Macro-F1": None, "n_gold": None, "n_pred": None})

COLS = ["Dataset", "Evaluation", "Span match", "P", "R", "F1", "Macro-F1", "n_gold", "n_pred"]
thesis_table = pd.DataFrame(rows)[COLS]

n_sum = results["exact"]["n_summaries"]
print(f"BERT+CRF event-trigger extraction (micro, span-based; values in %; N_TMA={n_sum} summaries)\n")
print(thesis_table.fillna("—").to_string(index=False))
print(
    "\nNote. TMA (Tell Me Again!) and MAVEN are NOT directly comparable. Micro, span-based; %. A predicted"
    "\ntrigger is correct for Identification if its sentence + character span matches a gold trigger, and"
    "\nfor Classification if the MAVEN type also matches (one-to-one; 'exact' = identical span, 'overlap'"
    f"\n= any character overlap). TMA: manual gold over N={n_sum} summaries, FREE extraction (no candidate"
    "\nlist / official negatives), so any predicted span absent from the gold is a false positive. MAVEN:"
    "\nin-domain (Wikipedia), candidate-based reference (Wang et al., 2020, Table 5; F1 67.8 ± 0.15, 10 runs)."
)

out_csv = GOLD_DIR / f"thesis_extraction_metrics.{VARIANT}.csv"
thesis_table.to_csv(out_csv, index=False)
print(f"\nsaved -> {out_csv.name}")


BERT+CRF event-trigger extraction (micro, span-based; values in %; N_TMA=0 summaries)

Dataset             Evaluation Span match    P    R   F1 Macro-F1 n_gold n_pred
    TMA Trigger identification      exact  0.0  0.0  0.0        —    0.0    0.0
    TMA Trigger classification      exact  0.0  0.0  0.0        —    0.0    0.0
    TMA Trigger identification    overlap  0.0  0.0  0.0        —    0.0    0.0
    TMA Trigger classification    overlap  0.0  0.0  0.0        —    0.0    0.0
  MAVEN Trigger classification      exact 65.0 70.9 67.8        —      —      —

Note. TMA (Tell Me Again!) and MAVEN are NOT directly comparable. Micro, span-based; %. A predicted
trigger is correct for Identification if its sentence + character span matches a gold trigger, and
for Classification if the MAVEN type also matches (one-to-one; 'exact' = identical span, 'overlap'
= any character overlap). TMA: manual gold over N=0 summaries, FREE extraction (no candidate
list / official negatives), so any predic